# PinkMap - Notebook 1: Collect Traffic Data

**What this does:** every time you run it, it asks Mapbox "how long does this trip take right now?" for 15 Houston trips, and adds 15 new rows to a spreadsheet file (`route_data.csv`) saved in your Google Drive.

**How to use it:**
1. Paste your Mapbox token in the first code cell.
2. Click **Runtime -> Run all**.
3. When Google asks to connect to your Drive, click Allow.
4. Run it again at different times of day and on different days. More runs = more data = a smarter model later.


## Step 1. Paste your Mapbox token
It starts with `pk.` Keep the quote marks around it.

In [ ]:
MAPBOX_TOKEN = "PASTE_YOUR_MAPBOX_PUBLIC_TOKEN_HERE"

if not MAPBOX_TOKEN.startswith("pk."):
    print("STOP: paste your real token between the quotes above (it starts with pk.)")
else:
    print("Token looks good.")

## Step 2. Get the tools ready and connect Google Drive
Your Drive is where the spreadsheet is saved, so it is still there tomorrow.

In [ ]:
import os
import requests
import pandas as pd
from datetime import datetime
from zoneinfo import ZoneInfo
from google.colab import drive

drive.mount("/content/drive")

SAVE_FOLDER = "/content/drive/MyDrive/PinkMap"
SAVE_FILE = SAVE_FOLDER + "/route_data.csv"
os.makedirs(SAVE_FOLDER, exist_ok=True)
print("Data will be saved to:", SAVE_FILE)

## Step 3. The list of trips
Each trip has a start place and an end place.

In [ ]:
# 15 trips between real Houston places (names and coordinates come from your team's OpenStreetMap list)
ROUTES = [
    {"route_id": 1, "origin": 'Houston City College', "o_lat": 29.74228, "o_lon": -95.3772125, "dest": 'Houston City College West Loop Campus', "d_lat": 29.7213883, "d_lon": -95.4573693},
    {"route_id": 2, "origin": 'Houston City College', "o_lat": 29.74228, "o_lon": -95.3772125, "dest": 'Houston City College Northeast - Northline Campus', "d_lat": 29.8328069, "d_lon": -95.3771485},
    {"route_id": 3, "origin": 'Houston City College', "o_lat": 29.74228, "o_lon": -95.3772125, "dest": 'Rice University', "d_lat": 29.7165599, "d_lon": -95.4029398},
    {"route_id": 4, "origin": 'Discovery Green', "o_lat": 29.7529825, "o_lon": -95.3595375, "dest": 'Hermann Park', "d_lat": 29.7148648, "d_lon": -95.3888866},
    {"route_id": 5, "origin": 'Discovery Green', "o_lat": 29.7529825, "o_lon": -95.3595375, "dest": 'University of Houston', "d_lat": 29.7207023, "d_lon": -95.3434964},
    {"route_id": 6, "origin": 'Houston City College West Loop Campus', "o_lat": 29.7213883, "o_lon": -95.4573693, "dest": 'Levy Park', "d_lat": 29.7326662, "d_lon": -95.4233311},
    {"route_id": 7, "origin": 'Houston City College West Loop Campus', "o_lat": 29.7213883, "o_lon": -95.4573693, "dest": 'Hermann Park', "d_lat": 29.7148648, "d_lon": -95.3888866},
    {"route_id": 8, "origin": 'Houston City College Northeast - Northline Campus', "o_lat": 29.8328069, "o_lon": -95.3771485, "dest": 'Discovery Green', "d_lat": 29.7529825, "d_lon": -95.3595375},
    {"route_id": 9, "origin": 'Memorial Park', "o_lat": 29.7661061, "o_lon": -95.4436138, "dest": 'Discovery Green', "d_lat": 29.7529825, "d_lon": -95.3595375},
    {"route_id": 10, "origin": 'Rice University', "o_lat": 29.7165599, "o_lon": -95.4029398, "dest": 'Space Center Houston', "d_lat": 29.5511263, "d_lon": -95.0973288},
    {"route_id": 11, "origin": 'Central Library', "o_lat": 29.7594337, "o_lon": -95.3699788, "dest": 'Houston Museum of Natural Science', "d_lat": 29.7221086, "d_lon": -95.3897295},
    {"route_id": 12, "origin": 'University of Houston', "o_lat": 29.7207023, "o_lon": -95.3434964, "dest": 'NRG Arena', "d_lat": 29.6811892, "d_lon": -95.4057955},
    {"route_id": 13, "origin": 'Houston City College Northwest - Spring Branch Campus', "o_lat": 29.7879064, "o_lon": -95.5606959, "dest": 'Houston City College', "d_lat": 29.74228, "d_lon": -95.3772125},
    {"route_id": 14, "origin": 'Houston City College West Loop Campus', "o_lat": 29.7213883, "o_lon": -95.4573693, "dest": 'Houston City College Northwest - Spring Branch Campus', "d_lat": 29.7879064, "d_lon": -95.5606959},
    {"route_id": 15, "origin": 'Houston Sports Park', "o_lat": 29.637277, "o_lon": -95.3950164, "dest": 'Discovery Green', "d_lat": 29.7529825, "d_lon": -95.3595375},
]

print(len(ROUTES), 'trips ready')

## Step 4. A small helper that asks Mapbox about one trip
It returns one row: the time, the distance, how long the trip takes with traffic, and how long it *normally* takes.

In [ ]:
def get_route_row(trip, token):
    coords = f"{trip['o_lon']},{trip['o_lat']};{trip['d_lon']},{trip['d_lat']}"
    url = f"https://api.mapbox.com/directions/v5/mapbox/driving-traffic/{coords}"
    params = {"access_token": token, "overview": "false", "alternatives": "false"}

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    route = response.json()["routes"][0]

    now = datetime.now(ZoneInfo("America/Chicago"))  # Houston time

    return {
        "collected_at": now.strftime("%Y-%m-%d %H:%M:%S"),
        "route_id": trip["route_id"],
        "origin": trip["origin"],
        "destination": trip["dest"],
        "distance_miles": round(route["distance"] / 1609.344, 3),
        "traffic_seconds": round(route["duration"], 1),
        "typical_seconds": route.get("duration_typical"),  # Mapbox's "normal day" time
        "day_of_week": now.strftime("%A"),
        "hour": now.hour,
    }

## Step 5. Ask Mapbox about all 15 trips and save the results

In [ ]:
rows = []
for trip in ROUTES:
    try:
        rows.append(get_route_row(trip, MAPBOX_TOKEN))
        print("OK   trip", trip["route_id"], trip["origin"], "->", trip["dest"])
    except Exception as error:
        print("FAIL trip", trip["route_id"], "-", error)

new_data = pd.DataFrame(rows)

if len(new_data) == 0:
    print("No rows were collected. Read the FAIL messages above.")
else:
    file_exists = os.path.exists(SAVE_FILE)
    new_data.to_csv(SAVE_FILE, mode="a", header=not file_exists, index=False)
    print(f"\nSaved {len(new_data)} new rows.")

## Step 6. Check your whole spreadsheet

In [ ]:
all_data = pd.read_csv(SAVE_FILE)
print("Total rows collected so far:", len(all_data))
print("Number of separate collection times:", all_data["collected_at"].str[:13].nunique())
all_data.tail(15)